In [1]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


In [3]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")

In [ ]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/train.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"

n = 60000
distance_threshold=0.01
use_mst=True
enrich=True

data_hetero_train, data_train = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
computing MST
MST done
enrichment
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [ ]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/test.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"
n = 15000
distance_threshold=0.01
use_mst=True
enrich=True

data_hetero_test, data_test = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
computing MST
MST done
enrichment
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN_model.HeteroTCR(data_hetero_train.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_train = data_hetero_train.to(device)
data_hetero_test = data_hetero_test.to(device)

In [7]:
# Create edge label indices for training data
train_edge_pairs = data_train[['tcr_index', 'epitope_index']].values
train_edge_label_index = torch.tensor(train_edge_pairs.T)

# Create edge label indices for test data
test_edge_pairs = data_test[['tcr_index', 'epitope_index']].values
test_edge_label_index = torch.tensor(test_edge_pairs.T)

# Create labels
y_train = data_train['Binding'].values
y_test = data_test['Binding'].values

In [8]:
import os

save_model_path = '/home/dsi/chenbis/repos/sol_lab/output/models'
NUM_EPOCHS=10

epoch = []
loss = []
acc = []
auc_roc = []
val_loss = []
val_acc = []
val_auc_roc = []
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0)

for ep in range(1, NUM_EPOCHS+1):
    train_loss, train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC = GNN_model.train(model, optimizer, 
                                                                                                                   data_hetero_train, data_hetero_test, 
                                                                                                                   train_edge_label_index, test_edge_label_index, 
                                                                                                                   y_train, y_test, 
                                                                                                                   device)
    
    print(
    'Train epoch: {} - loss: {:.4f} - binary_accuracy: {:.4f} - ROCAUC: {:.4f} - val_loss: {:.4f} - val_binary_accuracy: {:.4f} - val_ROCAUC: {:.4f}'.format(
    ep, train_loss.item(), train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC))
    torch.save(model.state_dict(), os.path.join(save_model_path, 'HeteroTCR_epoch{:03d}_AUC{:.6f}.pth'.format(ep, val_ROCAUC.item())))
    
    loss.append(train_loss.item())
    acc.append(train_binary_accuracy.item())
    auc_roc.append(train_ROCAUC.item())
    val_loss.append(valid_loss.item())
    val_acc.append(val_binary_accuracy.item())
    val_auc_roc.append(val_ROCAUC.item())
    epoch.append(ep)

Train epoch: 1 - loss: 0.6973 - binary_accuracy: 0.4830 - ROCAUC: 0.5438 - val_loss: 0.6992 - val_binary_accuracy: 0.5180 - val_ROCAUC: 0.4989
Train epoch: 2 - loss: 0.6980 - binary_accuracy: 0.5170 - ROCAUC: 0.5144 - val_loss: 0.6946 - val_binary_accuracy: 0.5180 - val_ROCAUC: 0.4647
Train epoch: 3 - loss: 0.6888 - binary_accuracy: 0.5230 - ROCAUC: 0.6046 - val_loss: 0.6961 - val_binary_accuracy: 0.4700 - val_ROCAUC: 0.4482
Train epoch: 4 - loss: 0.6872 - binary_accuracy: 0.5680 - ROCAUC: 0.6091 - val_loss: 0.6980 - val_binary_accuracy: 0.4760 - val_ROCAUC: 0.4427
Train epoch: 5 - loss: 0.6847 - binary_accuracy: 0.5740 - ROCAUC: 0.6147 - val_loss: 0.7010 - val_binary_accuracy: 0.5180 - val_ROCAUC: 0.4394
Train epoch: 6 - loss: 0.6813 - binary_accuracy: 0.5570 - ROCAUC: 0.6208 - val_loss: 0.7071 - val_binary_accuracy: 0.5120 - val_ROCAUC: 0.4404
Train epoch: 7 - loss: 0.6793 - binary_accuracy: 0.5440 - ROCAUC: 0.6203 - val_loss: 0.7100 - val_binary_accuracy: 0.5160 - val_ROCAUC: 0.4413

In [9]:
root_history = '/home/dsi/chenbis/repos/sol_lab/output'
# Logs of results
dfhistory = {'epoch': epoch,
            'loss': loss, 'acc': acc, 'auc_roc': auc_roc,
            'val_loss': val_loss, 'val_acc': val_acc, 'val_auc_roc': val_auc_roc}
df = pd.DataFrame(dfhistory)
if not os.path.exists(root_history):
    os.makedirs(root_history)
df.to_csv(f"{root_history}/history.csv", header=True, index=False)

# get num_epoch
val_auc_roc = list(df['val_auc_roc'])
max_auc_index = val_auc_roc.index(max(val_auc_roc))
num_epoch = max_auc_index + 1
print("max auc epoch:", num_epoch)
print(max(val_auc_roc))

val_loss = list(df['val_loss'])
min_val_loss_index = val_loss.index(min(val_loss))
loss_num_epoch = min_val_loss_index + 1
print("loss auc epoch:", loss_num_epoch)
print(val_auc_roc[min_val_loss_index])

for root, dirs, files in os.walk(save_model_path):
    for file in files:
        if file.startswith("HeteroTCR_epoch{:03d}".format(loss_num_epoch)):
            os.rename(os.path.join(save_model_path, file), os.path.join(save_model_path, 'minloss_AUC_' + file))
        elif file.startswith("HeteroTCR_epoch{:03d}".format(num_epoch)):
            os.rename(os.path.join(save_model_path, file), os.path.join(save_model_path, 'max_AUC_' + file))
        else:
            os.remove(os.path.join(save_model_path, file))

max auc epoch: 1
0.49893462657928467
loss auc epoch: 2
0.46469825506210327


In [10]:
import random

def generate_negative_edges(df, num_negatives, negative_score=0.0):
    """
    Efficiently generate negative cdr3~epitope pairs not present in df.
    """
    cdr3_pool = df['cdr3'].unique()
    epitope_pool = df['epitope'].unique()

    # Create full cartesian product
    all_combinations = pd.DataFrame(
        [(cdr3, epitope) for cdr3 in cdr3_pool for epitope in epitope_pool],
        columns=['cdr3', 'epitope']
    )

    # Remove positive (existing) pairs
    positives = df[['cdr3', 'epitope']].drop_duplicates()
    merged = all_combinations.merge(positives, on=['cdr3', 'epitope'], how='left', indicator=True)
    negatives = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

    # Shuffle and sample
    negatives = negatives.sample(n=min(num_negatives, len(negatives)), random_state=42).copy()
    negatives['score'] = negative_score
    negatives['source'] = 'NEGATIVE'

    # Truncate cdr3
    def truncate(seq):
        mid = (len(seq) + 1) // 2
        return seq[max(0, mid - 4):min(len(seq), mid + 4)]
    negatives['cdr3_truncated'] = negatives['cdr3'].apply(truncate)

    return negatives



def filter_data(df, source=None, min_score=None, max_score=None, num_negatives=0, negative_score=0.0):
    """
    Filters the combined DataFrame by source and score range, with optional negative edge generation.

    Parameters:
        df (pd.DataFrame): Combined dataset with 'source' and 'score' columns.
        source (str or list of str, optional): Filter by source(s) (e.g., "VDJDB" or ["McPAS", "TRAIT"]).
        min_score (float, optional): Minimum score (inclusive).
        max_score (float, optional): Maximum score (inclusive).
        num_negatives (int, optional): Number of negative cdr3~epitope pairs to generate.
        negative_score (float, optional): Score assigned to negative edges (default is 0.0).

    Returns:
        pd.DataFrame: Filtered DataFrame with optional negative edges appended.
    """
    filtered_df = df.copy()

    # Filter by source
    if source is not None:
        if isinstance(source, str):
            source = [source]
        filtered_df = filtered_df[filtered_df['source'].isin(source)]

    # Filter by score range
    if min_score is not None:
        filtered_df = filtered_df[filtered_df['score'] >= min_score]
    if max_score is not None:
        filtered_df = filtered_df[filtered_df['score'] <= max_score]

    # Efficient negative generation
    if num_negatives > 0:
        negative_df = generate_negative_edges(filtered_df, num_negatives, negative_score)
        filtered_df = pd.concat([filtered_df, negative_df], ignore_index=True)
    return filtered_df


In [ ]:
data_combined = pd.read_csv('/home/dsi/chenbis/repos/sol_lab/files/combined_data.csv')
data_combined = filter_data(data_combined, source=["McPAS"], num_negatives=10000)

cdr3_header = "cdr3"
label_header = "epitope"

n = len(data_combined)
distance_threshold=0.01
use_mst=True
enrich=True

combined_hetero, data_combined = create_hetero_data(data_combined, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)


sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
computing MST
MST done
enrichment
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [12]:

# Create edge label indices for test data
combined_edge_pairs = data_combined[['tcr_index', 'epitope_index']].values
combined_edge_label_index = torch.tensor(combined_edge_pairs.T)

# Create labels
y_combined = np.where(data_combined['source'] == 'NEGATIVE', 0, 1)

In [13]:
model = GNN_model.HeteroTCR(combined_hetero.metadata(), 1024, 3, "SAGE")
model = model.to(device)
combined_hetero = combined_hetero.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(combined_hetero.x_dict, combined_hetero.edge_index_dict, combined_edge_label_index)

model_dir = f"{root_history}/models"
val_model_path = f"{root_history}/models"
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, combined_hetero, combined_edge_label_index, y_combined)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

ACC: 0.5000
AUC: 0.1200


/tmp/ipykernel_149412/1542709223.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))


In [14]:
model = GNN_model.HeteroTCR(data_hetero_test.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_test = data_hetero_test.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_test.x_dict, data_hetero_test.edge_index_dict, test_edge_label_index)

model_dir = f"{root_history}/models"
val_model_path = f"{root_history}/models"
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, data_hetero_test, test_edge_label_index, y_test)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

ACC: 0.5180
AUC: 0.4647


/tmp/ipykernel_149412/856110204.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))
